# No-movement pulse matching notebook

This notebook compares two detector streams that observe the same radiation pulses:
- **Detector** (`dfd...`)
- **Reference** (`dfr...`)

Because both detectors are not moving, both should detect the same pulse train in time.
The workflow is:
1. Load CSV files into pandas DataFrames.
2. Detect pulses (`getpulses` / `getpulsesscan`).
3. Synchronize pulses (`syncpulses`).
4. Compute and visualize the ratio `ch0zcomplete / ch0zcomplete_reference`.


In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks, savgol_filter
import matplotlib.pyplot as plt

%matplotlib widget


## Load no-movement detector data

`skiprows` is used because the CSV includes a metadata header before the table.


In [ ]:
dfdall = pd.read_csv(
    "Output_Factor_6FFF_detector_no-movement-test_2_2026-04-19_14-35-36.csv",
    skiprows=35,
)
dfdall.head()


## Pulse extraction functions

### Current method (`getpulses`)
Main logic:
- Keep only `Time`, `ch0`, `ch1`.
- Estimate baseline (zero) from the first and last second.
- Subtract baseline (`ch0z`, `ch1z`).
- Build a threshold from a no-light region.
- Mark pulse samples and combine adjacent samples into `ch0zcomplete`.
- Keep one row per pulse in `single_pulse`.

`getpulsesscan` is the same logic but keeps scan coordinate `X`.


In [ ]:
def getpulses(df, threshold=0.2):
    """Detect pulses from a static (no-motion) acquisition DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with at least Time/ch0/ch1 columns.
    threshold : float
        Relative margin applied on top of no-light max (default 20%).

    Returns
    -------
    pd.DataFrame
        Pulse-aware dataframe containing baseline-corrected channels and
        one-row-per-pulse representation in `single_pulse`.
    """
    # Keep only columns used for pulse finding.
    df = df.loc[:, ["Time", "ch0", "ch1"]]

    # Baseline: first and last second are assumed to be no-light regions.
    lasttime = df.iloc[-1, 0]
    zeros = df.loc[(df.Time < 1) | (df.Time > (lasttime - 1)), ["ch0", "ch1"]].mean()

    # Baseline subtraction.
    dfzeros = df.loc[:, ["ch0", "ch1"]] - zeros
    dfzeros.columns = ["ch0z", "ch1z"]
    dfz = pd.concat([df, dfzeros], axis=1)

    # Approximate active pulse time window from rising edges in raw ch0.
    firstpulsetime = dfz.loc[dfz.ch0.diff() > 1, "Time"].min()
    lastpulsetime = dfz.loc[dfz.ch0.diff() > 1, "Time"].max()

    # Threshold = max no-light signal * (1 + margin).
    maxch0_nolight = dfz.loc[
        (dfz.Time < firstpulsetime - 0.2) | (dfz.Time > lastpulsetime + 0.2), "ch0z"
    ].max()
    maxch0_nolight_margin = maxch0_nolight * (1 + threshold)

    # Binary pulse mask.
    dfz["pulse"] = (dfz.ch0z > maxch0_nolight_margin).astype("boolean")

    # Pair adjacent pulse samples (many pulses are split over 2 rows).
    dfz["next_pulse"] = dfz.pulse.shift(-1)
    dfz["next_ch0z"] = dfz.ch0z.shift(-1)
    dfz["pulse_coincide"] = dfz.pulse & dfz.next_pulse
    dfz["pulse_coincide_after"] = dfz.pulse_coincide.shift()

    # Integrate adjacent pulse rows into one effective pulse value.
    dfz["ch0zcomplete"] = dfz.ch0z
    dfz.loc[dfz.pulse_coincide, "ch0zcomplete"] = dfz.ch0z + dfz.next_ch0z
    dfz.loc[dfz.pulse_coincide_after, "ch0zcomplete"] = 0

    # Keep only one row per pulse for matching/comparison.
    dfz["single_pulse"] = dfz.pulse & ~dfz.pulse_coincide_after

    return dfz.loc[
        :, ["Time", "ch0", "ch1", "ch0z", "ch1z", "single_pulse", "pulse_coincide", "ch0zcomplete"]
    ]


def getpulsesscan(df, threshold=0.2):
    """Same as getpulses(), preserving scan coordinate X."""
    df = df.loc[:, ["Time", "X", "ch0", "ch1"]]
    lasttime = df.iloc[-1, 0]
    zeros = df.loc[(df.Time < 1) | (df.Time > (lasttime - 1)), ["ch0", "ch1"]].mean()
    dfzeros = df.loc[:, ["ch0", "ch1"]] - zeros
    dfzeros.columns = ["ch0z", "ch1z"]
    dfz = pd.concat([df, dfzeros], axis=1)
    firstpulsetime = dfz.loc[dfz.ch0.diff() > 1, "Time"].min()
    lastpulsetime = dfz.loc[dfz.ch0.diff() > 1, "Time"].max()
    maxch0_nolight = dfz.loc[
        (dfz.Time < firstpulsetime - 0.2) | (dfz.Time > lastpulsetime + 0.2), "ch0z"
    ].max()
    maxch0_nolight_margin = maxch0_nolight * (1 + threshold)
    dfz["pulse"] = (dfz.ch0z > maxch0_nolight_margin).astype("boolean")
    dfz["next_pulse"] = dfz.pulse.shift(-1)
    dfz["next_ch0z"] = dfz.ch0z.shift(-1)
    dfz["pulse_coincide"] = dfz.pulse & dfz.next_pulse
    dfz["pulse_coincide_after"] = dfz.pulse_coincide.shift()
    dfz["ch0zcomplete"] = dfz.ch0z
    dfz.loc[dfz.pulse_coincide, "ch0zcomplete"] = dfz.ch0z + dfz.next_ch0z
    dfz.loc[dfz.pulse_coincide_after, "ch0zcomplete"] = 0
    dfz["single_pulse"] = dfz.pulse & ~dfz.pulse_coincide_after
    return dfz.loc[
        :, ["Time", "X", "ch0", "ch1", "ch0z", "ch1z", "single_pulse", "pulse_coincide", "ch0zcomplete"]
    ]


## Synchronization function

Current synchronization is index-based after filtering by `single_pulse`.
It works well **as long as both dataframes have exactly the same number of pulse rows**.


In [ ]:
def syncpulses(dfd, dfr):
    """Synchronize pulse values from reference dataframe into detector dataframe.

    Assumption: dfd.single_pulse.sum() == dfr.single_pulse.sum().
    """
    dft = dfd.copy()
    dft["single_pulse_reference"] = False
    dft["ch0zcomplete_reference"] = np.nan
    dft["pulse_coincide_reference"] = False

    mask = dft.single_pulse

    # This only works if there is the same number of single pulses in both data frames.
    results = dfr.loc[dfr.single_pulse, ["single_pulse", "ch0zcomplete", "pulse_coincide"]].to_numpy()

    dft.loc[mask, "single_pulse_reference"] = results[:, 0].astype(bool)
    dft.loc[mask, "ch0zcomplete_reference"] = results[:, 1].astype(float)
    dft.loc[mask, "pulse_coincide_reference"] = results[:, 2].astype(bool)
    dft["ratio"] = dft.ch0zcomplete / dft.ch0zcomplete_reference
    return dft


## Run no-movement comparison


In [ ]:
dfdp = getpulses(dfdall)

dfrall = pd.read_csv(
    "Output_Factor_6FFF_reference_no-movement-test_2_2026-04-19_14-35-29.csv",
    skiprows=33,
)
dfrp = getpulses(dfrall)

# Quality check: for no-movement acquisition, pulse counts should match.
same_pulse_count = dfrp.single_pulse.sum() == dfdp.single_pulse.sum()
print("Pulse counts equal:", same_pulse_count)
print("Detector pulses:", int(dfdp.single_pulse.sum()))
print("Reference pulses:", int(dfrp.single_pulse.sum()))

dft = syncpulses(dfdp, dfrp)
dft.head()


In [ ]:
ax1 = dft.plot.scatter(x="Time", y="ch0zcomplete", marker=".", alpha=0.2, label="detector")
dft.plot.scatter(
    x="Time",
    y="ch0zcomplete_reference",
    marker=".",
    color="red",
    ax=ax1,
    label="reference",
    alpha=0.2,
)
dft.plot.scatter(
    x="Time", y="ratio", marker=".", color="green", ax=ax1, label="ratio", alpha=0.2, grid=True
)
ax1.set_title("Detector and Reference No movement")
ax1.set_ylabel("Voltage in 30 pF capacitor (V)")
ax1.set_xlabel("Time (s)")


In [ ]:
# Quantify the visual observation: ratio has lower relative noise than each channel alone.
ratio_std = dft.loc[dft.single_pulse, "ratio"].std()
det_cv = dft.loc[dft.single_pulse, "ch0zcomplete"].std() / dft.loc[dft.single_pulse, "ch0zcomplete"].mean()
ref_cv = dft.loc[dft.single_pulse, "ch0zcomplete_reference"].std() / dft.loc[dft.single_pulse, "ch0zcomplete_reference"].mean()
ratio_cv = dft.loc[dft.single_pulse, "ratio"].std() / dft.loc[dft.single_pulse, "ratio"].mean()

print(f"ratio std: {ratio_std:.6f}")
print(f"detector CV: {det_cv:.6f}")
print(f"reference CV: {ref_cv:.6f}")
print(f"ratio CV: {ratio_cv:.6f}")


## Additional scan pair (5 mm/s)

This section verifies pulse extraction on one moving profile pair.


In [ ]:
dfd5 = pd.read_csv(
    "6FFF_detector_scan_air_10x10_6x_crossline_dmax_5mm-s_10.0x10.0_Profile_X_Depth-14.0_continuous_Speed-25.0_2026-04-19_15-01-45.csv",
    skiprows=43,
)

dfr5 = pd.read_csv(
    "Output_Factor_6FFF_reference_air_scan_10x10_6x_crossline_dmax_5mm-s_2026-04-19_15-01-39.csv",
    skiprows=33,
)

dfdp5 = getpulsesscan(dfd5)
dfrp5 = getpulses(dfr5)

print("Detector 5 mm/s pulses:", int(dfdp5.single_pulse.sum()))
print("Reference 5 mm/s pulses:", int(dfrp5.single_pulse.sum()))


In [ ]:
# Optional export for offline debugging / external scripts.
dfdp5.loc[:, ["Time", "single_pulse", "ch0zcomplete"]].to_csv("dfd.csv", index=False)
dfrp5.loc[:, ["Time", "single_pulse", "ch0zcomplete"]].to_csv("dfr.csv", index=False)


## Suggested improvements (pandas + SciPy)

Below are robust alternatives that can perform better when:
- baseline drifts,
- pulse amplitudes vary,
- pulse counts differ by a few missed detections,
- timestamps are slightly shifted.


In [ ]:
def getpulses_scipy(df, smooth_window=7, polyorder=2, sigma_threshold=5.0, prominence=0.01):
    """Alternative pulse detection using scipy.signal.find_peaks.

    Steps:
    1) Baseline subtraction from first/last second median.
    2) Optional Savitzky-Golay smoothing.
    3) Noise estimate from quiet regions.
    4) Peak detection with height + prominence constraints.
    """
    out = df.loc[:, ["Time", "ch0", "ch1"]].copy()
    lasttime = out.iloc[-1, 0]

    quiet = (out.Time < 1) | (out.Time > (lasttime - 1))
    baseline = out.loc[quiet, ["ch0", "ch1"]].median()
    out[["ch0z", "ch1z"]] = out[["ch0", "ch1"]] - baseline

    y = out["ch0z"].to_numpy()
    # window length for savgol must be odd and <= len(y)
    win = min(smooth_window if smooth_window % 2 == 1 else smooth_window + 1, len(y) - (len(y) + 1) % 2)
    y_smooth = savgol_filter(y, window_length=max(5, win), polyorder=min(polyorder, 3)) if len(y) >= 7 else y

    noise_std = np.nanstd(out.loc[quiet, "ch0z"])
    height = sigma_threshold * noise_std

    peaks, props = find_peaks(y_smooth, height=height, prominence=prominence)

    out["single_pulse"] = False
    out.loc[out.index[peaks], "single_pulse"] = True
    out["pulse_coincide"] = False
    out["ch0zcomplete"] = 0.0
    out.loc[out.single_pulse, "ch0zcomplete"] = out.loc[out.single_pulse, "ch0z"]

    return out.loc[:, ["Time", "ch0", "ch1", "ch0z", "ch1z", "single_pulse", "pulse_coincide", "ch0zcomplete"]]


def syncpulses_by_time(dfd, dfr, tolerance_s=0.002):
    """Alternative time-based synchronization using merge_asof.

    More robust than index alignment when some pulses are missing in one detector.
    """
    left = dfd.loc[dfd.single_pulse, ["Time", "ch0zcomplete"]].copy().sort_values("Time")
    right = dfr.loc[dfr.single_pulse, ["Time", "ch0zcomplete"]].copy().sort_values("Time")
    right = right.rename(columns={"Time": "Time_reference", "ch0zcomplete": "ch0zcomplete_reference"})

    merged = pd.merge_asof(
        left,
        right,
        left_on="Time",
        right_on="Time_reference",
        direction="nearest",
        tolerance=tolerance_s,
    )
    merged["ratio"] = merged["ch0zcomplete"] / merged["ch0zcomplete_reference"]
    return merged


### Notes on improvement usage
- Start by comparing pulse counts and matched-time residuals.
- If there are count mismatches, prefer `syncpulses_by_time` with a realistic tolerance.
- For noisy acquisitions, tune `sigma_threshold` and `prominence` in `getpulses_scipy`.
